# Module 01 — Lecture 2: The CUDA Programming Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/GPU-Programming-For-Computational-Neuroscience/blob/main/module_01_cuda_fundamentals/02_cuda_programming_model.ipynb)

---

In Lecture 1 we understood *why* GPUs are powerful. Now we learn *how to use them*: the CUDA programming model.

**Learning objectives:**
- Write a valid CUDA kernel using `__global__` and call it with `<<<grid, block>>>`
- Correctly compute global thread indices using built-in variables
- Understand host vs device memory and use `cudaMalloc`/`cudaMemcpy`
- Synchronize CPU and GPU with `cudaDeviceSynchronize`
- Choose appropriate block sizes for a given problem

In [ ]:
!nvidia-smi

## 1. CUDA Function Qualifiers

CUDA extends C++ with three function qualifiers:

| Qualifier | Runs on | Called from | Purpose |
|-----------|---------|-------------|--------|
| `__global__` | GPU | CPU (or GPU) | Kernel — the main entry point |
| `__device__` | GPU | GPU only | Helper function called from kernel |
| `__host__` | CPU | CPU only | Regular C++ function (default) |

```cpp
// A kernel: the function that thousands of GPU threads will run
__global__ void my_kernel(float* data, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;  // global index
    if (i < N) {
        data[i] = data[i] * 2.0f;   // each thread processes one element
    }
}

// A device helper: can only be called from __global__ or __device__ functions
__device__ float alpha_m(float V) {
    return 0.1f * (V + 40.0f) / (1.0f - expf(-(V + 40.0f) / 10.0f));
}
```

> **Rule:** If a function should run on the GPU, it must be `__global__` or `__device__`. Plain C++ functions run on the CPU only.

## 2. The Thread Hierarchy: Thread → Block → Grid

Every thread knows exactly where it is in the hierarchy through four built-in variables:

| Variable | Type | Meaning |
|----------|------|---------|
| `threadIdx.x` | uint3 | Thread index within its block |
| `blockIdx.x` | uint3 | Block index within the grid |
| `blockDim.x` | uint3 | Number of threads per block |
| `gridDim.x` | uint3 | Number of blocks in the grid |

### Computing the Global Index (1D case)

```
Block 0             Block 1             Block 2
┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│ T0 T1 T2 T3 │    │ T0 T1 T2 T3 │    │ T0 T1 T2 T3 │
│  0  1  2  3 │    │  4  5  6  7 │    │  8  9 10 11 │
└──────────────┘    └──────────────┘    └──────────────┘
  blockDim.x = 4

Global index: i = blockIdx.x * blockDim.x + threadIdx.x
  Block 1, Thread 2:  i = 1 * 4 + 2 = 6  ✓
```

This single formula maps every thread to a unique array element.

### Mapping to Neuroscience

```
Neuron array:  [0]  [1]  [2]  ...  [9999]
               ↑    ↑    ↑          ↑
Thread:        T0   T1   T2   ...  T9999
```

Thread `i` is responsible for neuron `i`. It reads `V[i]`, computes the update, writes back `V[i]`. No other thread touches `V[i]` — no conflicts.

In [ ]:
# Demonstrate thread indexing: print the hierarchy
%%writefile thread_index.cu
#include <stdio.h>

__global__ void show_indices() {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    printf("blockIdx=%d  threadIdx=%d  blockDim=%d  -->  global_i = %d\n",
           blockIdx.x, threadIdx.x, blockDim.x, i);
}

int main() {
    int threads_per_block = 4;
    int num_blocks = 3;
    printf("Launching %d blocks x %d threads:\n\n", num_blocks, threads_per_block);
    show_indices<<<num_blocks, threads_per_block>>>();
    cudaDeviceSynchronize();
    return 0;
}

In [ ]:
!nvcc -o thread_index thread_index.cu && ./thread_index

## 3. Host Memory vs Device Memory

The CPU and GPU have **separate memory spaces**. Data must be explicitly transferred.

```
HOST (CPU RAM)                    DEVICE (GPU VRAM)
──────────────                    ─────────────────
float* h_V    ──cudaMemcpy H→D──▶ float* d_V
              ◀─cudaMemcpy D→H──  (after kernel)

Allocation:
  CPU:  float* h_V = (float*)malloc(N * sizeof(float));
  GPU:  float* d_V;  cudaMalloc(&d_V, N * sizeof(float));

Deallocation:
  CPU:  free(h_V);
  GPU:  cudaFree(d_V);
```

**Convention:** Prefix host pointers with `h_` and device pointers with `d_`. This prevents costly mistakes (dereferencing a device pointer on the CPU causes a segfault).

### The Simulation Loop Pattern

```cpp
// 1. Allocate and initialize on CPU
float* h_V = malloc(N * sizeof(float));
init_neurons(h_V, N);

// 2. Copy to GPU (once)
float* d_V;
cudaMalloc(&d_V, N * sizeof(float));
cudaMemcpy(d_V, h_V, N * sizeof(float), cudaMemcpyHostToDevice);

// 3. Run simulation (many steps, data stays on GPU)
for (int t = 0; t < T_steps; t++) {
    update_neurons<<<blocks, threads>>>(d_V, d_I, N, dt);
}

// 4. Copy results back (once)
cudaMemcpy(h_V, d_V, N * sizeof(float), cudaMemcpyDeviceToHost);

// 5. Cleanup
cudaFree(d_V);
free(h_V);
```

Minimize CPU↔GPU transfers — they are slow (~10 GB/s PCIe vs 2000 GB/s GPU internal bandwidth). Keep data on the GPU for the entire simulation.

## 4. Choosing Block Size

How many threads per block? This is the most common configuration decision.

**Constraints:**
- Maximum 1024 threads per block (hardware limit)
- Block size must be a multiple of the warp size (32) for efficiency
- Shared memory and registers are divided among all threads in a block

**Practical rule:** Use **128, 256, or 512** threads per block. 256 is a safe default.

**Number of blocks:** Always use ceiling division to cover all elements:
```cpp
int threads = 256;
int blocks  = (N + threads - 1) / threads;   // ceiling division
```

**Always include a bounds check inside the kernel:**
```cpp
__global__ void my_kernel(float* data, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;    // extra threads in the last block do nothing
    data[i] = ...;
}
```

In [ ]:
# Complete example: scale membrane voltages (simulate a constant injected current step)
%%writefile scale_voltage.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do {                                          \
    cudaError_t e = (call);                                            \
    if (e != cudaSuccess) {                                            \
        fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(e));   \
        exit(1); }                                                     \
} while(0)

// Device helper: reversal potential shift
__device__ float apply_leak(float V, float E_L, float g_L, float dt) {
    return V + dt * (-g_L * (V - E_L));
}

// Kernel: update all neuron voltages with a leak current
__global__ void update_leak(float* V, float E_L, float g_L, float dt, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;                  // bounds check
    V[i] = apply_leak(V[i], E_L, g_L, dt);
}

int main() {
    const int N = 10000;       // 10,000 neurons
    const float dt  = 0.1f;   // ms
    const float E_L = -65.0f; // mV, leak reversal
    const float g_L = 0.01f;  // mS/cm^2, leak conductance

    size_t bytes = N * sizeof(float);

    // Host: initialize all neurons at -60 mV
    float* h_V = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) h_V[i] = -60.0f;

    // Device: allocate and copy
    float* d_V;
    CUDA_CHECK(cudaMalloc(&d_V, bytes));
    CUDA_CHECK(cudaMemcpy(d_V, h_V, bytes, cudaMemcpyHostToDevice));

    // Launch config
    int threads = 256;
    int blocks  = (N + threads - 1) / threads;
    printf("N=%d neurons, %d blocks x %d threads\n", N, blocks, threads);

    // Run 1000 steps (100 ms of simulated time)
    for (int t = 0; t < 1000; t++) {
        update_leak<<<blocks, threads>>>(d_V, E_L, g_L, dt, N);
    }
    CUDA_CHECK(cudaDeviceSynchronize());

    // Copy back and inspect
    CUDA_CHECK(cudaMemcpy(h_V, d_V, bytes, cudaMemcpyDeviceToHost));
    printf("After 100 ms, V[0] = %.4f mV  (expected ~%.4f mV)\n",
           h_V[0], E_L);

    cudaFree(d_V); free(h_V);
    return 0;
}

In [ ]:
!nvcc -O2 -o scale_voltage scale_voltage.cu && ./scale_voltage

## 5. Synchronization

CUDA kernel launches are **asynchronous** — control returns to the CPU immediately after `<<<>>>`. You must explicitly synchronize:

```cpp
my_kernel<<<blocks, threads>>>();    // returns immediately!
// ... CPU can do other work here ...
cudaDeviceSynchronize();             // CPU blocks until GPU finishes
cudaMemcpy(...);                     // safe to read results now
```

**Within a block:** use `__syncthreads()` to synchronize threads inside a block:
```cpp
__global__ void kernel(float* shared_input) {
    // Phase 1: all threads load data into shared memory
    __shared__ float smem[256];
    smem[threadIdx.x] = shared_input[...];   // load
    __syncthreads();                          // wait for all loads
    // Phase 2: now safe to read any smem[j]
    float result = smem[threadIdx.x] + smem[(threadIdx.x + 1) % 256];
}
```

There is **no synchronization across blocks** — blocks may run in any order and may not even be concurrent. Design algorithms so blocks are independent.

## 6. 2D Thread Grids (Bonus)

For 2D problems (e.g., weight matrices, 2D cortical fields), use 2D grids:

```cpp
dim3 threads_per_block(16, 16);                     // 16×16 = 256 threads
dim3 num_blocks((W + 15) / 16, (H + 15) / 16);     // cover W×H matrix
kernel_2d<<<num_blocks, threads_per_block>>>(d_mat, W, H);

__global__ void kernel_2d(float* mat, int W, int H) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;  // x → column
    int row = blockIdx.y * blockDim.y + threadIdx.y;  // y → row
    if (col >= W || row >= H) return;
    int idx = row * W + col;                           // row-major index
    mat[idx] = ...;
}
```

We will use this in Module 06 when processing 2D cortical grid simulations.

## Summary

| Concept | Syntax / Rule |
|---------|---------------|
| Kernel declaration | `__global__ void name(args...)` |
| Kernel launch | `name<<<num_blocks, threads_per_block>>>(args)` |
| Global index (1D) | `int i = blockIdx.x * blockDim.x + threadIdx.x;` |
| Bounds check | Always: `if (i >= N) return;` |
| Block size | Multiple of 32; use 128, 256, or 512 |
| GPU allocation | `cudaMalloc(&d_ptr, bytes)` |
| Host→GPU copy | `cudaMemcpy(d, h, bytes, cudaMemcpyHostToDevice)` |
| Synchronize | `cudaDeviceSynchronize()` before reading GPU results |

**Next lecture:** We write our first complete programs — `hello_cuda` and a fully timed vector addition — and compare performance to CPU.